# 🖊️ Task 1: Handwritten Text Generation
## Character-Level RNN (LSTM) – CodSoft AI/ML Internship

## 📌 Step 1: Environment Detection & Setup

In [ ]:
import os, sys

IS_KAGGLE = os.path.exists('/kaggle/input')
IS_COLAB  = 'google.colab' in sys.modules
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB

print(f"Running on: {'Kaggle' if IS_KAGGLE else 'Google Colab' if IS_COLAB else 'Local Machine'}")

if IS_KAGGLE:
    DATA_DIR = '/kaggle/input/english-handwritten-characters-dataset'
else:
    DATA_DIR = 'dataset'

CSV_PATH = os.path.join(DATA_DIR, 'english.csv')
IMG_DIR  = os.path.join(DATA_DIR, 'Img')

os.makedirs('models', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
print(f'CSV Path: {CSV_PATH}')
print(f'Image Dir: {IMG_DIR}')

## 📥 Step 2: Dataset Download (Skip on Kaggle)

In [ ]:
if not IS_KAGGLE:
    if IS_COLAB:
        print('On Google Colab - please upload your kaggle.json')
        from google.colab import files
        uploaded = files.upload()
        os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
        import shutil
        shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
        os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

    if not os.path.exists(CSV_PATH):
        print('Downloading dataset from Kaggle...')
        os.system('pip install kaggle -q')
        os.makedirs(DATA_DIR, exist_ok=True)
        os.system(
            f'kaggle datasets download -d dhruvildave/english-handwritten-characters-dataset '
            f'-p {DATA_DIR} --unzip'
        )
        print('Dataset downloaded!')
    else:
        print('Dataset already exists locally.')
else:
    print('Kaggle environment - dataset already mounted.')
print('Dataset check done.')

## 📦 Step 3: Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install',
     'tensorflow', 'numpy', 'pandas', 'matplotlib',
     'Pillow', 'opencv-python', 'scikit-learn', 'seaborn', 'tqdm', '-q'],
    capture_output=True,
)
print('All packages ready!')

## 📚 Step 4: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import random
import json
import warnings
warnings.filterwarnings('ignore')

from PIL import Image
from tqdm import tqdm
from collections import Counter

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# GPU Memory Management
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU found: {[g.name for g in gpus]}')
else:
    print('No GPU found - using CPU')
print(f'TensorFlow: {tf.__version__}')

## 📊 Step 5: Load & Explore Dataset

In [ ]:
df = pd.read_csv(CSV_PATH)
if len(df.columns) == 2:
    df.columns = ['image', 'label']
elif 'label' not in df.columns:
    df.rename(columns={df.columns[-1]: 'label', df.columns[0]: 'image'}, inplace=True)

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(df.head(10))
print(f'\nTotal samples: {len(df)}')
print(f'Unique characters: {df["label"].nunique()}')
print(f'Labels: {sorted(df["label"].unique())}')

## 📈 Step 6: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

label_counts = df['label'].value_counts().sort_index()
axes[0].bar(range(len(label_counts)), label_counts.values, color='steelblue', edgecolor='white')
axes[0].set_xticks(range(len(label_counts)))
axes[0].set_xticklabels(label_counts.index, fontsize=7)
axes[0].set_title('Character Label Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Character')
axes[0].set_ylabel('Count')

labels_list = label_counts.index.tolist()
digits = [l for l in labels_list if str(l).isdigit()]
upper  = [l for l in labels_list if str(l).isupper()]
lower  = [l for l in labels_list if str(l).islower()]
cat_counts = [len(digits), len(upper), len(lower)]
axes[1].pie(
    cat_counts,
    labels=[f'Digits ({len(digits)})', f'Uppercase ({len(upper)})', f'Lowercase ({len(lower)})'],
    autopct='%1.1f%%',
    colors=['#FF6B6B', '#4ECDC4', '#45B7D1'],
    startangle=90,
)
axes[1].set_title('Character Categories', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 🖼️ Step 7: Sample Handwritten Character Images

In [ ]:
def load_image(img_filename, img_dir, size=(64, 64)):
    path = os.path.join(img_dir, img_filename)
    if not os.path.exists(path):
        return None
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    return cv2.resize(img, size)

unique_labels = sorted(df['label'].unique())[:30]
fig, axes = plt.subplots(3, 10, figsize=(20, 6))
axes = axes.flatten()

shown = 0
for lbl in unique_labels:
    subset = df[df['label'] == lbl]
    img = load_image(subset.iloc[0]['image'], IMG_DIR)
    if img is not None:
        axes[shown].imshow(img, cmap='gray')
        axes[shown].set_title(str(lbl), fontsize=12, fontweight='bold')
        axes[shown].axis('off')
        shown += 1
    if shown >= 30:
        break

for i in range(shown, len(axes)):
    axes[i].axis('off')

plt.suptitle('Sample Handwritten Characters', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/sample_characters.png', dpi=150, bbox_inches='tight')
plt.show()

## 🗂️ Step 8: Build Character Image Bank

In [ ]:
image_bank = {}
for _, row in tqdm(df.iterrows(), total=len(df), desc='Building image bank'):
    char = str(row['label'])
    fname = row['image']
    if char not in image_bank:
        image_bank[char] = []
    image_bank[char].append(fname)

print(f'Image bank: {len(image_bank)} unique characters')
for k, v in sorted(image_bank.items()):
    print(f"  '{k}' -> {len(v)} images")

## 🔤 Step 9: Build Character Vocabulary & Text Corpus

In [ ]:
corpus = list(df['label'].astype(str).values)
print(f'Corpus length: {len(corpus)}')

vocab = sorted(set(corpus))
vocab_size = len(vocab)
print(f'Vocabulary size: {vocab_size}')
print(f'Vocabulary: {vocab}')

char2idx = {c: i for i, c in enumerate(vocab)}
idx2char  = {i: c for c, i in char2idx.items()}

corpus_encoded = [char2idx[c] for c in corpus]
print(f'\nFirst 20 encoded: {corpus_encoded[:20]}')
print(f'First 20 chars:   {corpus[:20]}')

## 🔢 Step 10: Create Training Sequences

In [ ]:
SEQ_LENGTH = 40
STEP       = 3
BATCH_SIZE = 256
EPOCHS     = 50
EMBED_DIM  = 64
LSTM_UNITS = 256

X_seqs, y_seqs = [], []
for i in range(0, len(corpus_encoded) - SEQ_LENGTH, STEP):
    X_seqs.append(corpus_encoded[i : i + SEQ_LENGTH])
    y_seqs.append(corpus_encoded[i + SEQ_LENGTH])

X_seqs = np.array(X_seqs)
y_seqs = np.array(y_seqs)

print(f'X shape: {X_seqs.shape}')
print(f'y shape: {y_seqs.shape}')
print(f'Training sequences: {len(X_seqs):,}')

y_onehot = to_categorical(y_seqs, num_classes=vocab_size)
print(f'y one-hot shape: {y_onehot.shape}')

X_train, X_val, y_train, y_val = train_test_split(X_seqs, y_onehot, test_size=0.1, random_state=42)
print(f'Train: {X_train.shape}, Val: {X_val.shape}')

## 🧠 Step 11: Build the Character-Level RNN Model

In [ ]:
def build_char_rnn(vocab_size, seq_length, embed_dim=64, lstm_units=256):
    """
    Stacked LSTM Character-Level Language Model.
    Architecture optimized for RTX 3050 (6GB) and Colab T4 (15GB).
    """
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embed_dim,
                  input_length=seq_length, name='embedding'),
        LSTM(lstm_units, return_sequences=True, name='lstm_1'),
        Dropout(0.3, name='dropout_1'),
        LSTM(lstm_units, return_sequences=True, name='lstm_2'),
        Dropout(0.3, name='dropout_2'),
        LSTM(lstm_units // 2, return_sequences=False, name='lstm_3'),
        Dropout(0.2, name='dropout_3'),
        Dense(128, activation='relu', name='dense_hidden'),
        BatchNormalization(name='batch_norm'),
        Dense(vocab_size, activation='softmax', name='output'),
    ], name='CharRNN')
    return model

model = build_char_rnn(vocab_size, SEQ_LENGTH, EMBED_DIM, LSTM_UNITS)
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()
print(f'Total parameters: {model.count_params():,}')
print(f'Estimated model size: {model.count_params() * 4 / 1e6:.1f} MB')

## ⚙️ Step 12: Training Callbacks

In [ ]:
callbacks = [
    ModelCheckpoint(
        filepath='models/char_rnn_best.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1,
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1,
    ),
]
print('Callbacks configured.')

## 🚀 Step 13: Train the Model

In [ ]:
# Adjust batch size based on GPU memory
if gpus:
    # RTX 3050: 6GB -> 256, Colab T4: 15GB -> 512
    effective_batch = BATCH_SIZE
else:
    effective_batch = 64  # CPU fallback

print(f'Training with batch size: {effective_batch}')
print(f'Train samples: {len(X_train):,}')
print(f'Val samples:   {len(X_val):,}')
print(f'Max epochs:    {EPOCHS}')
print('=' * 50)

history = model.fit(
    X_train, y_train,
    batch_size=effective_batch,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1,
)

print('\n Training complete!')
print(f'Best val accuracy: {max(history.history["val_accuracy"]):.4f}')
print(f'Best val loss:     {min(history.history["val_loss"]):.4f}')

## 📉 Step 14: Training & Validation Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(history.history['loss'],     label='Train Loss',       color='#2196F3', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss',  color='#F44336', linewidth=2, linestyle='--')
axes[0].set_title('Loss Curve', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='Train Accuracy',      color='#4CAF50', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', color='#FF9800', linewidth=2, linestyle='--')
axes[1].set_title('Accuracy Curve', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Final train accuracy: {history.history["accuracy"][-1]:.4f}')
print(f'Final val accuracy:   {history.history["val_accuracy"][-1]:.4f}')

## 🌡️ Step 15: Temperature Sampling & Text Generation

In [ ]:
def sample_with_temperature(predictions, temperature=1.0):
    """
    Sample next character index using temperature scaling.
    T < 1.0 = conservative (repetitive), T > 1.0 = creative (diverse).
    """
    predictions = np.asarray(predictions).astype('float64')
    predictions = np.log(predictions + 1e-8) / temperature
    exp_preds   = np.exp(predictions - np.max(predictions))
    predictions = exp_preds / np.sum(exp_preds)
    return np.argmax(np.random.multinomial(1, predictions, 1))


def generate_text(model, seed_chars, num_generate=100, temperature=1.0):
    """
    Generate character sequence from seed using trained RNN.
    """
    seed = [c for c in seed_chars if c in char2idx]
    if len(seed) < SEQ_LENGTH:
        pad  = [random.choice(list(char2idx.keys())) for _ in range(SEQ_LENGTH - len(seed))]
        seed = pad + seed
    seed = seed[-SEQ_LENGTH:]

    generated   = list(seed)
    current_seq = [char2idx[c] for c in seed]

    for _ in range(num_generate):
        x         = np.array(current_seq[-SEQ_LENGTH:]).reshape(1, SEQ_LENGTH)
        preds     = model.predict(x, verbose=0)[0]
        next_idx  = sample_with_temperature(preds, temperature)
        next_char = idx2char[next_idx]
        generated.append(next_char)
        current_seq.append(next_idx)

    return generated[SEQ_LENGTH:]

print('Generation functions ready.')

## ✨ Step 16: Generate Handwritten-Like Text Sequences

In [ ]:
NUM_CHARS     = 80
seed_sequence = corpus[:SEQ_LENGTH]
print(f"Seed: {''.join(seed_sequence)}")
print('=' * 60)

temperatures     = [0.5, 0.8, 1.0, 1.2, 1.5]
generated_results = {}

for temp in temperatures:
    generated        = generate_text(model, seed_sequence, num_generate=NUM_CHARS, temperature=temp)
    generated_str    = ''.join(generated)
    generated_results[temp] = generated
    print(f'\n Temperature = {temp}:')
    print(f'   {generated_str}')
    print(f'   Unique chars: {len(set(generated))} | Length: {len(generated)}')

with open('outputs/generated_text.txt', 'w') as f:
    f.write(f"Seed: {''.join(seed_sequence)}\n\n")
    for temp, gen in generated_results.items():
        f.write(f'Temperature {temp}:\n')
        f.write(''.join(gen) + '\n\n')

print('\n Generated text saved to outputs/generated_text.txt')

## 🖼️ Step 17: Render Generated Text as Handwritten Images

In [ ]:
def render_text_as_handwritten(char_sequence, image_bank, img_dir, char_size=64, padding=4):
    """
    Stitch real handwritten character images together to form a visual sentence.
    """
    char_images = []
    for char in char_sequence:
        if char in image_bank and image_bank[char]:
            fname = random.choice(image_bank[char])
            img   = load_image(fname, img_dir, size=(char_size, char_size))
            char_images.append(
                img if img is not None else np.full((char_size, char_size), 255, dtype=np.uint8)
            )
        else:
            char_images.append(np.full((char_size, char_size), 255, dtype=np.uint8))

    if not char_images:
        return None

    pad_col = np.full((char_size, padding), 255, dtype=np.uint8)
    row = char_images[0]
    for img in char_images[1:]:
        row = np.concatenate([row, pad_col, img], axis=1)
    return row


fig, axes = plt.subplots(len(temperatures), 1, figsize=(20, 4 * len(temperatures)))

for i, temp in enumerate(temperatures):
    gen_seq  = generated_results[temp][:20]
    rendered = render_text_as_handwritten(gen_seq, image_bank, IMG_DIR)
    if rendered is not None:
        axes[i].imshow(rendered, cmap='gray', aspect='auto')
        axes[i].set_title(
            f" Temperature = {temp}  |  Generated: {''.join(gen_seq)}",
            fontsize=12, fontweight='bold', pad=8,
        )
    else:
        axes[i].text(0.5, 0.5, f"Generated: {''.join(gen_seq)}",
                     transform=axes[i].transAxes, ha='center', fontsize=12)
    axes[i].axis('off')

plt.suptitle('Generated Handwritten Text (Real Character Images from Dataset)',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/generated_handwritten_text.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Saved to outputs/generated_handwritten_text.png')

## 📊 Step 18: Character Frequency Analysis

In [ ]:
fig, axes = plt.subplots(1, len(temperatures), figsize=(20, 4))

for i, temp in enumerate(temperatures):
    gen    = generated_results[temp]
    freq   = Counter(gen)
    chars  = sorted(freq.keys())
    counts = [freq[c] for c in chars]
    axes[i].bar(chars, counts, color=plt.cm.viridis(i / len(temperatures)))
    axes[i].set_title(f'T={temp}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Character')
    if i == 0:
        axes[i].set_ylabel('Frequency')
    axes[i].tick_params(axis='x', labelsize=7, rotation=45)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Generated Character Frequency by Temperature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/char_frequency_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 🎮 Step 19: Interactive Text Generation

In [ ]:
def interactive_generate(seed_text, length=60, temperature=1.0):
    seed      = list(str(seed_text))
    generated = generate_text(model, seed, num_generate=length, temperature=temperature)
    result    = ''.join(generated)
    print(f"\n Seed: {''.join(seed)}")
    print(f" Generated ({length} chars, T={temperature}):")
    print(f"   {result}")

    rendered = render_text_as_handwritten(generated[:30], image_bank, IMG_DIR)
    if rendered is not None:
        plt.figure(figsize=(18, 3))
        plt.imshow(rendered, cmap='gray')
        plt.title(f'Handwritten render: {result[:30]}...', fontsize=12)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    return result

interactive_generate(['a', 'b', 'c'], length=50, temperature=0.8)
interactive_generate(['1', '2', '3'], length=50, temperature=1.0)
interactive_generate(['A', 'B', 'C'], length=50, temperature=1.2)

## 💾 Step 20: Save Model & Metadata

In [ ]:
model.save('models/char_rnn_final.h5')
print(' Model saved to models/char_rnn_final.h5')

vocab_data = {
    'vocab'      : vocab,
    'char2idx'   : char2idx,
    'idx2char'   : {str(k): v for k, v in idx2char.items()},
    'seq_length' : SEQ_LENGTH,
    'vocab_size' : vocab_size,
}
with open('models/vocab.json', 'w') as f:
    json.dump(vocab_data, f, indent=2)
print(' Vocabulary saved to models/vocab.json')

print('\n' + '=' * 60)
print(' TRAINING SUMMARY')
print('=' * 60)
print(f'Model:             Character-Level LSTM RNN')
print(f'Vocab size:        {vocab_size}')
print(f'Sequence length:   {SEQ_LENGTH}')
print(f'Training samples:  {len(X_train):,}')
print(f'Total parameters:  {model.count_params():,}')
print(f'Best val accuracy: {max(history.history["val_accuracy"]):.4f}')
print(f'Best val loss:     {min(history.history["val_loss"]):.4f}')
print('=' * 60)
print(' Task 1 Complete!')

## 🎉 Task 1 Complete!

| Step | Description | Status |
|------|-------------|--------|
| 1 | Environment Detection | ✅ |
| 2 | Dataset Download | ✅ |
| 3 | Dependencies | ✅ |
| 4 | Imports | ✅ |
| 5 | Data Loading | ✅ |
| 6 | EDA | ✅ |
| 7 | Sample Images | ✅ |
| 8 | Image Bank | ✅ |
| 9 | Vocabulary | ✅ |
| 10 | Sequences | ✅ |
| 11 | Model Building | ✅ |
| 12 | Callbacks | ✅ |
| 13 | Training | ✅ |
| 14 | Training Curves | ✅ |
| 15 | Temperature Sampling | ✅ |
| 16 | Text Generation | ✅ |
| 17 | Image Rendering | ✅ |
| 18 | Analysis | ✅ |
| 19 | Interactive Gen | ✅ |
| 20 | Save Model | ✅ |

> **CodSoft Internship – Task 1: Handwritten Text Generation Complete** 🖊️